[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/44_dr_grpo_loss_solution.ipynb)

# 参考解答：去偏组相对策略优化损失（Dr. GRPO）

参考解答。

In [ ]:
# 在 Colab 中安装 torch-judge（JupyterLab/Docker 中会跳过）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F
from torch import Tensor

In [ ]:
# ✅ 参考解答

def dr_grpo_loss(new_logps: Tensor,
                 old_logps: Tensor,
                 rewards: Tensor,
                 group_ids: Tensor,
                 mask: Tensor | None = None,
                 clip_ratio: float = 0.2,
                 max_tokens: int | None = None) -> Tensor:
    """去偏组相对策略优化（Dr. GRPO）损失。"""
    r_detached = rewards.detach().to(dtype=new_logps.dtype, device=new_logps.device)
    advantages = torch.zeros_like(r_detached)

    for gid in group_ids.unique():
        g_mask = group_ids == gid
        r_g = r_detached[g_mask]
        advantages[g_mask] = r_g - r_g.mean()

    if mask is None:
        token_mask = torch.ones_like(new_logps)
    else:
        token_mask = mask.to(dtype=new_logps.dtype, device=new_logps.device)

    log_diff = torch.where(
        token_mask > 0,
        new_logps - old_logps.detach(),
        torch.zeros_like(new_logps),
    )
    ratios = torch.exp(log_diff)
    adv_2d = advantages.detach().unsqueeze(-1)

    unclipped = ratios * adv_2d
    clipped = torch.clamp(ratios, 1.0 - clip_ratio, 1.0 + clip_ratio) * adv_2d
    surrogate = torch.min(unclipped, clipped)

    batch_size, seq_len = new_logps.shape
    l_max = float(max_tokens if max_tokens is not None else seq_len)
    return -(surrogate * token_mask).sum() / (batch_size * l_max)

In [ ]:
# 示例
ratios = torch.tensor([[1.35, 1.10], [0.65, 0.90]])
old_logps = torch.zeros(2, 2)
new_logps = ratios.log().requires_grad_(True)
rewards = torch.tensor([1.0, -1.0])
group_ids = torch.tensor([0, 0])
print('Loss:', dr_grpo_loss(new_logps, old_logps, rewards, group_ids, max_tokens=2).item())

In [ ]:
from torch_judge import check
check('dr_grpo_loss')